# 🏛️ RBI & SEBI Financial Regulatory RAG Pipeline
## Model Variant: `llama3:latest` (Llama 3 (8B))

This self-contained Jupyter Notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline specifically designed for **Indian Financial Regulations** (Reserve Bank of India - **RBI** and Securities and Exchange Board of India - **SEBI** circulars, master directions, and notifications).

---

### 🏗️ Complete RAG Architecture
```
PDF Documents (RBI / SEBI Guidelines)
        │
        ▼
Regulatory Structure-Aware Chunking ───► Preserves document hierarchy, sections & metadata
        │
    ┌───┴────────────────────────┐
    ▼                            ▼
Dense Retrieval             Sparse Retrieval
(ChromaDB + BGE-Small)       (BM25 with regulatory tokenizer)
    │                            │
    └─────────────┬──────────────┘
                  ▼
      Reciprocal Rank Fusion (RRF)
                  ▼
      Cross-Encoder Reranking
      (cross-encoder/ms-marco-MiniLM-L-6-v2)
                  ▼
           Ollama Local LLM
           (llama3:latest)
                  ▼
     Grounded Answer + Citations
                  ▼
           Discrete Claims
                  ▼
      Automated RAG Evaluation
```

---

### 📋 Notebook Roadmap
1. **Dependencies & Imports**: Load libraries for parsing, vector storage, BM25, reranking, and Ollama HTTP integration.
2. **Configuration**: Hyperparameters, model paths, chunk sizes, and retrieval constants.
3. **Ollama Health Check**: Test connection to local Ollama instance and verify model `llama3:latest`.
4. **Ingestion & Parsing**: PyMuPDF-based text extraction with header/footer cleanup.
5. **Structure-Aware Chunking**: Regulatory chunker creating contextual breadcrumbs (`[Issuer | Doc | Chapter | Section | Page]`).
6. **Indexing (ChromaDB + BM25)**: Build or load dense vector and sparse lexical indexes.
7. **Hybrid RRF & Cross-Encoder Reranking**: Multi-stage candidate retrieval and deep cross-attention reranking.
8. **Grounded Generation & Claims Separation**: Structured prompting with citation grounding and atomic claims extraction.
9. **Dedicated Test Query**: Complete pipeline execution for *"What is the Cash Reserve Ratio requirement?"*.
10. **Benchmark Evaluation Suite**: Automated evaluation measuring Hit Rate@K, MRR@K, Precision@K, and Citation Coverage.


In [6]:
# ==============================================================================
# Cell 2: [OPTIONAL] Install Required Dependencies
# Uncomment and execute if running in an environment where packages are not pre-installed.
# ==============================================================================
%pip install -q pymupdf sentence-transformers chromadb rank_bm25 requests pandas numpy scikit-learn tqdm


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install requests


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
     -------------------------------------- 131.1/131.1 kB 3.9 MB/s eta 0:00:00
     -------------------------------------- 206.0/206.0 kB 4.2 MB/s eta 0:00:00
     -------------------------------------- 137.0/137.0 kB 7.9 MB/s eta 0:00:00
     ---------------------------------------- 68.5/68.5 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ==============================================================================
# Cell 3: Import Standard and Third-Party Libraries
# ==============================================================================
import os
import sys
import re
import json
import time
import pickle
import requests
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Document Parsing & Storage
import pymupdf  # PyMuPDF for high-fidelity PDF extraction
import chromadb

# Dense Embeddings & Cross-Encoder Reranking
from sentence_transformers import SentenceTransformer, CrossEncoder

# Sparse Lexical Retrieval
from rank_bm25 import BM25Okapi

print("[OK] All libraries successfully imported.")


c:\Users\laksh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] All libraries successfully imported.


In [2]:
# ==============================================================================
# Cell 4: System Configuration & Hyperparameters
# ==============================================================================
# Base Working Directories
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "RBI-GUIDELINES"
PERSIST_DIRECTORY = BASE_DIR / ".chroma_db"
SPARSE_INDEX_PATH = BASE_DIR / ".bm25_index.pkl"
EVAL_DATASET_PATH = BASE_DIR / "data" / "eval_dataset.json"

# Embedding & Cross-Encoder Models (CPU-Optimized)
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"             # 384-dimensional dense embeddings
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2" # Lightweight cross-encoder

# Regulatory Structure Chunking Hyperparameters
TARGET_CHUNK_SIZE = 600   # Target token/word length per chunk
CHUNK_OVERLAP = 100       # Overlap lines/words across chunk boundaries
MAX_CHUNK_SIZE = 1000     # Hard ceiling for chunks

# Multi-Stage Retrieval Hyperparameters
TOP_K_DENSE = 15          # Candidate chunks from ChromaDB dense search
TOP_K_SPARSE = 15         # Candidate chunks from BM25 sparse search
TOP_K_HYBRID = 15         # Candidate pool after Reciprocal Rank Fusion
TOP_K_RERANKED = 5        # Final reranked context chunks passed to LLM
RRF_K = 60                # Reciprocal Rank Fusion constant

print(f"Base Directory:     {BASE_DIR}")
print(f"Dataset Directory:  {DATASET_DIR}")
print(f"ChromaDB Path:      {PERSIST_DIRECTORY}")
print(f"BM25 Index Path:    {SPARSE_INDEX_PATH}")
print(f"Embedding Model:    {EMBEDDING_MODEL_NAME}")
print(f"Reranker Model:     {RERANKER_MODEL_NAME}")


Base Directory:     c:\Users\laksh\Downloads\FYP_RAG
Dataset Directory:  c:\Users\laksh\Downloads\FYP_RAG\RBI-GUIDELINES
ChromaDB Path:      c:\Users\laksh\Downloads\FYP_RAG\.chroma_db
BM25 Index Path:    c:\Users\laksh\Downloads\FYP_RAG\.bm25_index.pkl
Embedding Model:    BAAI/bge-small-en-v1.5
Reranker Model:     cross-encoder/ms-marco-MiniLM-L-6-v2


In [3]:
# ==============================================================================
# Cell 5: Ollama Model & Endpoint Configuration
# ==============================================================================
OLLAMA_MODEL = "llama3:latest"
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_TIMEOUT = 3600  # 1 hour timeout (3600s) for local CPU/GPU inference

print(f"Configured Ollama Model: {OLLAMA_MODEL}")
print(f"Configured Ollama Host:  {OLLAMA_HOST}")
print(f"Ollama Request Timeout:  {OLLAMA_TIMEOUT}s (1 hour)")


Configured Ollama Model: llama3:latest
Configured Ollama Host:  http://localhost:11434
Ollama Request Timeout:  3600s (1 hour)


In [4]:
# ==============================================================================
# Cell 6: Verify Ollama Service & Model Availability
# ==============================================================================
def check_ollama_service(host: str, target_model: str) -> bool:
    """Pings the local Ollama HTTP API and verifies that the configured model is ready."""
    clean_host = host.rstrip('/')
    tags_url = f"{clean_host}/api/tags"
    
    print(f"Checking Ollama server at {tags_url}...")
    try:
        response = requests.get(tags_url, timeout=10)
        if response.status_code != 200:
            print(f"[ERROR] Ollama returned HTTP status code {response.status_code}.")
            return False
            
        payload = response.json()
        models = [m.get("name", "") for m in payload.get("models", [])]
        print(f"[OK] Ollama is active and reachable.")
        print(f"Installed local models: {models}")
        
        # Check if the target model is installed
        model_found = any(m == target_model or m.startswith(target_model.split(':')[0]) for m in models)
        if model_found:
            print(f"[SUCCESS] Target model '{target_model}' is ready for generation!")
            return True
        else:
            print(f"[WARNING] Model '{target_model}' is not in the installed models list.")
            print(f"Please run this command in your terminal to download it:")
            print(f"    ollama pull {target_model}")
            return False
            
    except requests.exceptions.ConnectionError:
        print(f"[ERROR] Connection to Ollama at {clean_host} failed.")
        print("Ensure the Ollama application or background service is running on your machine.")
        return False
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred while contacting Ollama: {e}")
        return False

# Execute check
ollama_service_active = check_ollama_service(OLLAMA_HOST, OLLAMA_MODEL)


Checking Ollama server at http://localhost:11434/api/tags...
[OK] Ollama is active and reachable.
Installed local models: ['qwen2.5:7b-instruct-q4_K_M', 'llama3:latest', 'gemma3:4b']
[SUCCESS] Target model 'llama3:latest' is ready for generation!


### 📄 Ingestion Pipeline: Dataset Scanner & Regulatory PDF Parser
- **DatasetScanner**: Traverses the guidelines directory, detects document issuer (`RBI` vs `SEBI`), infers document category (`Master Circular`, `Master Direction`, `Act`, `Regulation`, `Rule`, `Circular`, etc.), and cleans title naming.
- **RegulatoryPDFParser**: Uses PyMuPDF to extract text page-by-page while stripping standalone header/footer noise (such as page counters).


In [5]:
# ==============================================================================
# Cell 7: Ingestion Components (DatasetScanner & RegulatoryPDFParser)
# ==============================================================================
class DatasetScanner:
    """Discovers, categorizes, and extracts high-level metadata from regulatory PDF files."""

    def __init__(self, root_dir: Path):
        self.root_dir = Path(root_dir)

    def scan_documents(self, max_docs: Optional[int] = None) -> List[Dict[str, Any]]:
        """Walks dataset directory and returns structured document descriptors."""
        documents = []

        if not self.root_dir.exists():
            print(f"[Warning] Dataset directory not found: {self.root_dir}")
            return documents

        for root, _, files in os.walk(self.root_dir):
            pdf_files = [f for f in files if f.lower().endswith(".pdf")]
            for filename in pdf_files:
                file_path = Path(root) / filename
                metadata = self._extract_file_metadata(file_path)
                documents.append(metadata)

                if max_docs and len(documents) >= max_docs:
                    return documents

        print(f"[Ingestion] Scanned {len(documents)} PDF documents from {self.root_dir}")
        return documents

    def _extract_file_metadata(self, file_path: Path) -> Dict[str, Any]:
        path_str = str(file_path)
        filename = file_path.name
        try:
            rel_path = file_path.relative_to(self.root_dir)
        except Exception:
            rel_path = filename

        # Determine Issuer (RBI vs SEBI)
        if "sebi" in path_str.lower():
            issuer = "SEBI"
        else:
            issuer = "RBI"

        # Determine Regulatory Category
        category = "Guideline"
        lower_path = path_str.lower()
        if "master_circular" in lower_path or "master circular" in lower_path:
            category = "Master Circular"
        elif "master_direction" in lower_path or "master direction" in lower_path:
            category = "Master Direction"
        elif "act" in lower_path or "acts" in lower_path:
            category = "Act"
        elif "regulation" in lower_path or "regulations" in lower_path:
            category = "Regulation"
        elif "rule" in lower_path or "rules" in lower_path:
            category = "Rule"
        elif "gazette" in lower_path:
            category = "Gazette Notification"
        elif "circular" in lower_path:
            category = "Circular"

        # Extract Year from filename or path structure
        year_match = re.search(r'(19\d{2}|20\d{2})', path_str)
        year = year_match.group(1) if year_match else "Unknown"

        # Clean Document Title
        doc_title = file_path.stem
        clean_title = re.sub(r'[_\-]+', ' ', doc_title).strip()

        return {
            "doc_id": f"{issuer}_{doc_title}".replace(" ", "_"),
            "file_path": str(file_path),
            "filename": filename,
            "relative_path": str(rel_path),
            "doc_title": clean_title,
            "issuer": issuer,
            "category": category,
            "year": year
        }


class RegulatoryPDFParser:
    """Parses regulatory PDF documents into page-level structured text blocks with headers."""

    def parse_pdf(self, file_path: str) -> List[Dict[str, Any]]:
        """Extracts text, page numbers, and structural headings from a PDF file."""
        pages = []

        try:
            doc = pymupdf.open(file_path)
            for page_num in range(len(doc)):
                page = doc[page_num]
                text = page.get_text("text")

                # Clean header/footer artifacts
                cleaned_text = self._clean_page_text(text)

                if cleaned_text.strip():
                    pages.append({
                        "page_number": page_num + 1,
                        "text": cleaned_text,
                        "raw_text": text
                    })

            doc.close()
        except Exception as e:
            print(f"[Error] Failed to parse PDF {file_path}: {e}")

        return pages

    def _clean_page_text(self, text: str) -> str:
        """Removes repeated header/footer noise while preserving section structure."""
        lines = text.splitlines()
        cleaned_lines = []

        for line in lines:
            stripped = line.strip()
            # Ignore standalone page numbers (e.g., 'Page 1 of 12' or isolated digits)
            if re.match(r'^(page\s+\d+(\s+of\s+\d+)?|\d+)$', stripped, re.IGNORECASE):
                continue
            cleaned_lines.append(line)

        return "\n".join(cleaned_lines)

print("[OK] DatasetScanner and RegulatoryPDFParser defined.")


[OK] DatasetScanner and RegulatoryPDFParser defined.


### ✂️ Regulatory Structure-Aware Chunking
Preserves hierarchical context by recognizing Chapters, Parts, Sections, and Paragraphs. Each chunk is augmented with an explicit **breadcrumb string** (`[Issuer: ... | Category: ... | Year: ... | Doc: ... | Chapter: ... | Section: ... | Page: ...]`) to ensure full semantic grounding during embedding and retrieval.


In [6]:
# ==============================================================================
# Cell 8: Structure-Aware Chunking & Breadcrumbs
# ==============================================================================
@dataclass
class RegulatoryChunk:
    chunk_id: str
    doc_id: str
    text: str
    breadcrumb_text: str
    metadata: Dict[str, Any]


class RegulatoryStructureChunker:
    """Structure-aware chunker tailored for RBI & SEBI regulatory guidelines."""

    def __init__(self, target_chunk_size: int = TARGET_CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
        self.target_chunk_size = target_chunk_size
        self.overlap = overlap

    def chunk_document(self, doc_metadata: Dict[str, Any], pages: List[Dict[str, Any]]) -> List[RegulatoryChunk]:
        """Splits page text into structure-aware chunks with contextual breadcrumbs."""
        chunks: List[RegulatoryChunk] = []
        if not pages:
            return chunks

        doc_title = doc_metadata.get("doc_title", "Regulatory Document")
        issuer = doc_metadata.get("issuer", "Regulatory Authority")
        category = doc_metadata.get("category", "Guideline")
        year = doc_metadata.get("year", "Unknown")

        current_chapter = "General"
        current_section = "Main Provisions"

        # Regex patterns for regulatory headings
        chapter_pattern = re.compile(r'^(chapter|part)\s+([IVXLCDM\d]+[:\.\-]?.*)', re.IGNORECASE)
        section_pattern = re.compile(r'^(\d+[\.\d]*\s+[A-Z].*|section\s+\d+.*|paragraph\s+\d+.*)', re.IGNORECASE)

        chunk_counter = 0

        for page in pages:
            page_num = page["page_number"]
            text = page["text"]
            lines = text.split("\n")

            buffer_lines = []
            buffer_word_count = 0

            for line in lines:
                stripped = line.strip()
                if not stripped:
                    continue

                # Check for chapter update
                chap_match = chapter_pattern.match(stripped)
                if chap_match:
                    current_chapter = stripped[:80]

                # Check for section update
                sec_match = section_pattern.match(stripped)
                if sec_match:
                    current_section = stripped[:100]

                buffer_lines.append(stripped)
                buffer_word_count += len(stripped.split())

                # Flush buffer when target chunk size is reached
                if buffer_word_count >= self.target_chunk_size:
                    chunk_text = "\n".join(buffer_lines)
                    chunk_counter += 1

                    breadcrumb = f"[Issuer: {issuer} | Category: {category} | Year: {year} | Doc: {doc_title} | Chapter: {current_chapter} | Section: {current_section} | Page: {page_num}]"
                    full_text_with_breadcrumb = f"{breadcrumb}\n\n{chunk_text}"

                    meta = {
                        "doc_id": doc_metadata["doc_id"],
                        "file_path": doc_metadata["file_path"],
                        "issuer": issuer,
                        "category": category,
                        "year": year,
                        "doc_title": doc_title,
                        "chapter": current_chapter,
                        "section": current_section,
                        "page_number": page_num,
                        "chunk_index": chunk_counter
                    }

                    chunks.append(RegulatoryChunk(
                        chunk_id=f"{doc_metadata['doc_id']}_chunk_{chunk_counter}",
                        doc_id=doc_metadata["doc_id"],
                        text=chunk_text,
                        breadcrumb_text=full_text_with_breadcrumb,
                        metadata=meta
                    ))

                    # Retain last few lines for overlap
                    overlap_lines = buffer_lines[-3:] if len(buffer_lines) > 3 else []
                    buffer_lines = list(overlap_lines)
                    buffer_word_count = sum(len(l.split()) for l in buffer_lines)

            # Flush remaining page buffer
            if buffer_lines:
                chunk_text = "\n".join(buffer_lines)
                chunk_counter += 1

                breadcrumb = f"[Issuer: {issuer} | Category: {category} | Year: {year} | Doc: {doc_title} | Chapter: {current_chapter} | Section: {current_section} | Page: {page_num}]"
                full_text_with_breadcrumb = f"{breadcrumb}\n\n{chunk_text}"

                meta = {
                    "doc_id": doc_metadata["doc_id"],
                    "file_path": doc_metadata["file_path"],
                    "issuer": issuer,
                    "category": category,
                    "year": year,
                    "doc_title": doc_title,
                    "chapter": current_chapter,
                    "section": current_section,
                    "page_number": page_num,
                    "chunk_index": chunk_counter
                }

                chunks.append(RegulatoryChunk(
                    chunk_id=f"{doc_metadata['doc_id']}_chunk_{chunk_counter}",
                    doc_id=doc_metadata["doc_id"],
                    text=chunk_text,
                    breadcrumb_text=full_text_with_breadcrumb,
                    metadata=meta
                ))

        return chunks

print("[OK] RegulatoryChunk and RegulatoryStructureChunker defined.")


[OK] RegulatoryChunk and RegulatoryStructureChunker defined.


### 🧠 Dense Vector Store: ChromaDB with BGE-Small Embeddings
- Utilizes `BAAI/bge-small-en-v1.5` for fast 384-dimensional vector embeddings on CPU.
- Uses BGE query instruction: `"Represent this sentence for searching relevant passages: {query}"`.


In [7]:
# ==============================================================================
# Cell 9: ChromaDB Dense Vector Store
# ==============================================================================
class ChromaVectorStore:
    """Persistent ChromaDB vector store backed by BAAI/bge-small-en-v1.5 embeddings."""

    def __init__(self, persist_dir: Path = PERSIST_DIRECTORY, model_name: str = EMBEDDING_MODEL_NAME):
        self.persist_dir = Path(persist_dir)
        self.model_name = model_name

        self.persist_dir.mkdir(parents=True, exist_ok=True)

        print(f"[VectorStore] Loading embedding model: {model_name}...")
        self.embedder = SentenceTransformer(model_name, device="cpu")

        print(f"[VectorStore] Initializing ChromaDB client at {self.persist_dir}...")
        self.client = chromadb.PersistentClient(path=str(self.persist_dir))
        self.collection = self.client.get_or_create_collection(
            name="rbi_sebi_guidelines",
            metadata={"hnsw:space": "cosine"}
        )

    def add_chunks(self, chunks: List[RegulatoryChunk], batch_size: int = 100):
        """Indexes regulatory chunks into ChromaDB with embeddings and metadata."""
        if not chunks:
            return

        print(f"[VectorStore] Indexing {len(chunks)} chunks into ChromaDB...")
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            ids = [c.chunk_id for c in batch]
            texts = [c.breadcrumb_text for c in batch]

            embeddings = self.embedder.encode(texts, show_progress_bar=False, convert_to_numpy=True).tolist()
            metadatas = [c.metadata for c in batch]

            self.collection.upsert(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas
            )

        print(f"[VectorStore] Index complete! Total documents in ChromaDB collection: {self.collection.count()}")

    def similarity_search(self, query: str, top_k: int = TOP_K_DENSE, filter_dict: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """Performs dense vector similarity search with query prefix."""
        count = self.collection.count()
        if count == 0:
            return []

        query_embedding = self.embedder.encode(
            [f"Represent this sentence for searching relevant passages: {query}"],
            convert_to_numpy=True
        ).tolist()

        kwargs = {
            "query_embeddings": query_embedding,
            "n_results": min(top_k, max(1, count))
        }
        if filter_dict:
            kwargs["where"] = filter_dict

        results = self.collection.query(**kwargs)

        search_hits = []
        if results and results.get("ids") and results["ids"][0]:
            ids = results["ids"][0]
            docs = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0] if "distances" in results else [0.0] * len(ids)

            for cid, doc, meta, dist in zip(ids, docs, metadatas, distances):
                similarity = 1.0 - float(dist)
                search_hits.append({
                    "chunk_id": cid,
                    "text": doc,
                    "metadata": meta,
                    "score": similarity
                })

        return search_hits

    def count(self) -> int:
        return self.collection.count()

print("[OK] ChromaVectorStore defined.")


[OK] ChromaVectorStore defined.


### 🔍 Sparse Lexical Retriever: BM25 with Regulatory Tokenizer
- Features a domain-adapted regex tokenizer (`regulatory_tokenize`) preserving clause numbers, circular identifiers, percentages, acronyms, and section codes.


In [8]:
# ==============================================================================
# Cell 10: BM25 Sparse Lexical Retriever
# ==============================================================================
def regulatory_tokenize(text: str) -> List[str]:
    """Custom tokenizer preserving section IDs, circular refs, percentages, and acronyms."""
    if not text:
        return []
    tokens = re.findall(r'\b[A-Za-z0-9\-\.\/\%]+\b', text.lower())
    return [t for t in tokens if len(t) > 1]


class BM25Retriever:
    """Sparse BM25 retriever for exact clause, term, and numerical matching."""

    def __init__(self, index_path: Optional[Path] = SPARSE_INDEX_PATH):
        self.index_path = Path(index_path) if index_path else None
        self.bm25: Optional[BM25Okapi] = None
        self.chunks: List[RegulatoryChunk] = []
        self.tokenized_corpus: List[List[str]] = []

    def build_index(self, chunks: List[RegulatoryChunk]):
        """Builds BM25 index over regulatory chunks."""
        print(f"[BM25] Tokenizing {len(chunks)} chunks for sparse index...")
        self.chunks = chunks
        self.tokenized_corpus = [regulatory_tokenize(c.breadcrumb_text) for c in chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        if self.index_path:
            self.save_index(self.index_path)

    def search(self, query: str, top_k: int = TOP_K_SPARSE) -> List[Dict[str, Any]]:
        """Searches BM25 index and returns top-k ranked chunks with scores."""
        if not self.bm25 or not self.chunks:
            print("[Warning] BM25 index is empty.")
            return []

        query_tokens = regulatory_tokenize(query)
        if not query_tokens:
            return []

        scores = self.bm25.get_scores(query_tokens)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        results = []
        for idx in top_indices:
            score = float(scores[idx])
            chunk = self.chunks[idx]
            results.append({
                "chunk_id": chunk.chunk_id,
                "text": chunk.breadcrumb_text,
                "metadata": chunk.metadata,
                "score": score
            })
        return results

    def save_index(self, file_path: Path):
        """Saves BM25 index to pickle file."""
        file_path = Path(file_path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(file_path, "wb") as f:
            pickle.dump({"chunks": self.chunks, "tokenized_corpus": self.tokenized_corpus}, f)
        print(f"[BM25] Saved sparse index with {len(self.chunks)} chunks to {file_path}")

    def load_index(self, file_path: Path) -> bool:
        """Loads BM25 index from pickle file if exists."""
        file_path = Path(file_path)
        if not file_path.exists():
            return False

        try:
            with open(file_path, "rb") as f:
                data = pickle.load(f)
                self.chunks = data["chunks"]
                self.tokenized_corpus = data["tokenized_corpus"]
                self.bm25 = BM25Okapi(self.tokenized_corpus)
            print(f"[BM25] Loaded sparse index with {len(self.chunks)} chunks from {file_path}")
            return True
        except Exception as e:
            print(f"[Error] Failed to load BM25 index from {file_path}: {e}")
            return False

print("[OK] BM25Retriever and regulatory_tokenize defined.")


[OK] BM25Retriever and regulatory_tokenize defined.


### 🔨 Indexing Pipeline - Option A: [BUILD INDEX]
> **Use this cell if you need to build or rebuild the ChromaDB and BM25 indexes from the `RBI-GUIDELINES/` folder.**
> If indexes already exist on disk, you can skip this cell and run Option B below.


In [9]:
# ==============================================================================
# Cell 11: [OPTION A - BUILD INDEX] Scan PDFs, Generate Chunks, Build Indexes
# ==============================================================================
def build_full_index(dataset_dir: Path = DATASET_DIR, max_docs: Optional[int] = None):
    """Parses PDFs from the dataset directory and creates both Dense (ChromaDB) and Sparse (BM25) indexes."""
    print("=== [BUILD INDEX] INGESTION & DUAL INDEXING PIPELINE ===")
    scanner = DatasetScanner(root_dir=dataset_dir)
    doc_descriptors = scanner.scan_documents(max_docs=max_docs)

    if not doc_descriptors:
        print(f"[Error] No PDF files found in {dataset_dir}")
        return None, None, []

    parser = RegulatoryPDFParser()
    chunker = RegulatoryStructureChunker(target_chunk_size=TARGET_CHUNK_SIZE, overlap=CHUNK_OVERLAP)

    all_chunks = []
    print(f"\n[Ingestion] Parsing {len(doc_descriptors)} regulatory PDF documents...")

    for idx, doc_meta in enumerate(doc_descriptors, start=1):
        print(f"[{idx}/{len(doc_descriptors)}] Parsing: {doc_meta['filename']}")
        pages = parser.parse_pdf(doc_meta["file_path"])
        chunks = chunker.chunk_document(doc_meta, pages)
        all_chunks.extend(chunks)

    print(f"\n[Ingestion] Total structure-aware chunks generated: {len(all_chunks)}")

    # 1. Build ChromaDB Vector Store
    vector_store = ChromaVectorStore(persist_dir=PERSIST_DIRECTORY, model_name=EMBEDDING_MODEL_NAME)
    vector_store.add_chunks(all_chunks)

    # 2. Build BM25 Sparse Index
    bm25_retriever = BM25Retriever(index_path=SPARSE_INDEX_PATH)
    bm25_retriever.build_index(all_chunks)

    print("\n[SUCCESS] Ingestion & dual indexing complete!")
    return vector_store, bm25_retriever, all_chunks

# UNCOMMENT TO RUN FULL INDEXING:
# vector_store, bm25_retriever, all_chunks = build_full_index()


### 📂 Indexing Pipeline - Option B: [LOAD EXISTING INDEX]
> **Use this cell to load the existing `.chroma_db` vector database and `.bm25_index.pkl` sparse index without re-parsing PDFs.**


In [10]:
# ==============================================================================
# Cell 12: [OPTION B - LOAD INDEX] Load Existing ChromaDB and BM25 Index
# ==============================================================================
def load_existing_indexes() -> Tuple[ChromaVectorStore, BM25Retriever]:
    """Loads existing ChromaDB vector store and BM25 sparse index from disk."""
    print("=== [LOAD INDEX] Initializing Vector Store & BM25 Retriever ===")
    
    # 1. Initialize Vector Store
    vector_store = ChromaVectorStore(persist_dir=PERSIST_DIRECTORY, model_name=EMBEDDING_MODEL_NAME)
    doc_count = vector_store.count()
    print(f"[VectorStore] Active documents in ChromaDB collection: {doc_count}")

    # 2. Initialize and Load BM25 Retriever
    bm25_retriever = BM25Retriever(index_path=SPARSE_INDEX_PATH)
    loaded = bm25_retriever.load_index(SPARSE_INDEX_PATH)
    if not loaded:
        print("[Warning] BM25 index not found. If this is a fresh setup, run Option A (Cell 11) to build index.")

    return vector_store, bm25_retriever

# Load the indexes
vector_store, bm25_retriever = load_existing_indexes()


=== [LOAD INDEX] Initializing Vector Store & BM25 Retriever ===
[VectorStore] Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4129.23it/s]


[VectorStore] Initializing ChromaDB client at c:\Users\laksh\Downloads\FYP_RAG\.chroma_db...
[VectorStore] Active documents in ChromaDB collection: 3898
[BM25] Loaded sparse index with 76 chunks from c:\Users\laksh\Downloads\FYP_RAG\.bm25_index.pkl


### ⚡ Hybrid Retrieval (Dense + Sparse RRF) & Cross-Encoder Reranking
1. **Dense Semantic Search**: ChromaDB retrieves Top-$K_{dense}$ hits.
2. **Sparse Lexical Search**: BM25 retrieves Top-$K_{sparse}$ hits.
3. **Reciprocal Rank Fusion (RRF)**: Merges rank positions using $RRF(d) = \sum \frac{1}{60 + \text{rank} + 1}$.
4. **Cross-Encoder Reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2` performs joint attention scoring over `(query, context)` pairs to select the top 5 most relevant contexts.


In [11]:
# ==============================================================================
# Cell 13: Hybrid Retriever & Cross-Encoder Reranker
# ==============================================================================
class HybridRerankRetriever:
    """Hybrid Retriever combining Dense Vector + Sparse BM25 search with Cross-Encoder reranking."""

    def __init__(
        self,
        vector_store: ChromaVectorStore,
        bm25_retriever: BM25Retriever,
        reranker_model_name: str = RERANKER_MODEL_NAME,
        rrf_k: int = RRF_K
    ):
        self.vector_store = vector_store
        self.bm25_retriever = bm25_retriever
        self.rrf_k = rrf_k
        self.reranker_model_name = reranker_model_name

        print(f"[Reranker] Loading Cross-Encoder model: {reranker_model_name}...")
        self.cross_encoder = CrossEncoder(reranker_model_name, device="cpu")

    def retrieve(
        self,
        query: str,
        top_k_dense: int = TOP_K_DENSE,
        top_k_sparse: int = TOP_K_SPARSE,
        top_k_final: int = TOP_K_RERANKED,
        filter_dict: Optional[Dict[str, Any]] = None
    ) -> List[Dict[str, Any]]:
        """Executes Hybrid Dense + Sparse RRF search followed by Cross-Encoder reranking."""
        # 1. Dense Semantic Search
        dense_hits = self.vector_store.similarity_search(query, top_k=top_k_dense, filter_dict=filter_dict)

        # 2. Sparse Lexical Search
        sparse_hits = self.bm25_retriever.search(query, top_k=top_k_sparse)

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores: Dict[str, float] = {}
        chunk_map: Dict[str, Dict[str, Any]] = {}

        for rank, hit in enumerate(dense_hits):
            cid = hit["chunk_id"]
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank + 1)
            chunk_map[cid] = hit

        for rank, hit in enumerate(sparse_hits):
            cid = hit["chunk_id"]
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank + 1)
            if cid not in chunk_map:
                chunk_map[cid] = hit

        if not rrf_scores:
            print("[Warning] No candidates returned from Dense or Sparse retrieval.")
            return []

        # Sort candidate pool by RRF score
        sorted_candidates = sorted(rrf_scores.keys(), key=lambda cid: rrf_scores[cid], reverse=True)
        candidate_chunks = [chunk_map[cid] for cid in sorted_candidates[:top_k_dense + top_k_sparse]]

        # 4. Cross-Encoder Reranking
        pairs = [[query, chunk["text"]] for chunk in candidate_chunks]
        cross_scores = self.cross_encoder.predict(pairs, show_progress_bar=False)

        for chunk, c_score in zip(candidate_chunks, cross_scores):
            chunk["rerank_score"] = float(c_score)

        reranked_chunks = sorted(candidate_chunks, key=lambda c: c["rerank_score"], reverse=True)
        final_top_k = reranked_chunks[:top_k_final]

        return final_top_k

# Initialize Hybrid Retriever
hybrid_retriever = HybridRerankRetriever(
    vector_store=vector_store,
    bm25_retriever=bm25_retriever,
    reranker_model_name=RERANKER_MODEL_NAME,
    rrf_k=RRF_K
)
print("[OK] HybridRerankRetriever initialized.")


[Reranker] Loading Cross-Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4377.75it/s]


[OK] HybridRerankRetriever initialized.


### 🤖 Ollama Grounded Generator & Claims Separation
- Calls the local Ollama HTTP API endpoint (`/api/generate`) with `model="llama3:latest"` and `stream=False`.
- Formats retrieved contexts into `[Doc (Year), Section]` citation stamps.
- Prompts the LLM to output factual answers grounded exclusively in the context, ending with a structured `DISCRETE_CLAIMS:` block.
- **Important**: Does NOT silently switch to an extractive CPU fallback if Ollama fails. Instead, clearly raises and displays the exact Ollama error for easy debugging.


In [12]:
# ==============================================================================
# Cell 14: RegulatoryGenerator (Ollama HTTP Integration)
# ==============================================================================
class RegulatoryGenerator:
    """Generation module for regulatory QA with explicit section citations and SLM claim extraction."""

    def __init__(self, model_name: str = OLLAMA_MODEL, ollama_host: str = OLLAMA_HOST, timeout: int = OLLAMA_TIMEOUT):
        self.model_name = model_name
        self.ollama_host = ollama_host.rstrip('/')
        self.timeout = timeout

    def generate_answer(self, query: str, context_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Synthesizes answer with citations and discrete claims from retrieved context chunks."""
        if not context_chunks:
            return {
                "answer": "No relevant RBI/SEBI regulatory context was found to answer your query.",
                "citations": [],
                "claims": ["No relevant regulatory guidelines available."],
                "model_used": self.model_name
            }

        # Format context block and extract citation tags
        formatted_contexts = []
        citations = []

        for idx, chunk in enumerate(context_chunks, start=1):
            meta = chunk.get("metadata", {})
            doc = meta.get("doc_title", "Doc")
            sec = meta.get("section", "Section")
            yr = meta.get("year", "")
            citation_str = f"[{doc} ({yr}), {sec}]"
            if citation_str not in citations:
                citations.append(citation_str)

            text_snippet = chunk.get("text", "")
            formatted_contexts.append(f"Context [{idx}] {citation_str}:\n{text_snippet}\n")

        context_str = "\n".join(formatted_contexts)

        # Build System & User Prompts
        system_prompt = (
            "You are an expert RBI and SEBI financial regulatory compliance officer. "
            "Answer the user's question using ONLY the provided regulatory context snippets. "
            "You must cite document names, sections, or clauses. "
            "At the end of your response, output a section titled 'DISCRETE_CLAIMS:' listing "
            "each factual assertion as an atomic bullet point for validation by an SLM verifier."
        )

        user_prompt = (
            f"Query: {query}\n\n"
            f"Retrieved Regulatory Context:\n{context_str}\n\n"
            f"Provide a thorough, grounded answer with citations followed by DISCRETE_CLAIMS."
        )

        # Call Ollama local service
        ollama_response = self._call_ollama(system_prompt, user_prompt)
        answer_text, claims = self._parse_claims(ollama_response)

        return {
            "answer": answer_text,
            "citations": citations,
            "claims": claims,
            "model_used": self.model_name
        }

    def _call_ollama(self, system_prompt: str, user_prompt: str) -> str:
        """Calls local Ollama API server. Raises RuntimeError on connection/model errors."""
        url = f"{self.ollama_host}/api/generate"
        payload = {
            "model": self.model_name,
            "prompt": f"{system_prompt}\n\n{user_prompt}",
            "stream": False,
            "options": {
                "temperature": 0.1,
                "num_ctx": 4096
            }
        }

        try:
            resp = requests.post(url, json=payload, timeout=self.timeout)
            if resp.status_code == 200:
                data = resp.json()
                return data.get("response", "")
            else:
                error_msg = f"Ollama HTTP Error [{resp.status_code}]: {resp.text}"
                print(f"[Generator ERROR] {error_msg}")
                raise RuntimeError(error_msg)
        except requests.exceptions.ConnectionError as e:
            error_msg = f"Failed to connect to Ollama at {self.ollama_host}. Is Ollama running? Error: {e}"
            print(f"[Generator ERROR] {error_msg}")
            raise ConnectionError(error_msg) from e
        except requests.exceptions.Timeout as e:
            error_msg = f"Ollama request timed out after {self.timeout} seconds."
            print(f"[Generator ERROR] {error_msg}")
            raise TimeoutError(error_msg) from e
        except Exception as e:
            print(f"[Generator ERROR] Ollama API error: {e}")
            raise

    def _parse_claims(self, raw_response: str) -> Tuple[str, List[str]]:
        """Extracts the main answer text and discrete claims list from raw LLM output."""
        if "DISCRETE_CLAIMS:" in raw_response:
            parts = raw_response.split("DISCRETE_CLAIMS:")
            answer_text = parts[0].strip()
            claims_block = parts[1].strip()
            claims = [re.sub(r'^[\-\*\d\.\s]+', '', line).strip() for line in claims_block.splitlines() if line.strip()]
        else:
            answer_text = raw_response.strip()
            sentences = [s.strip() for s in re.split(r'[\.\!\?]\s+', answer_text) if len(s.strip()) > 15]
            claims = sentences[:5]

        return answer_text, claims if claims else ["Generated claim supported by retrieved regulatory text."]

# Initialize Generator
generator = RegulatoryGenerator(model_name=OLLAMA_MODEL, ollama_host=OLLAMA_HOST, timeout=OLLAMA_TIMEOUT)
print(f"[OK] RegulatoryGenerator initialized with model: {OLLAMA_MODEL}")


[OK] RegulatoryGenerator initialized with model: llama3:latest


### 🔎 Complete Pipeline Query Function
Combines Hybrid RRF Retrieval, Cross-Encoder Reranking, and Grounded Ollama Generation.


In [13]:
# ==============================================================================
# Cell 15: Complete Pipeline Query Function
# ==============================================================================
def run_regulatory_rag_query(query: str, verbose: bool = True) -> Dict[str, Any]:
    """Executes full end-to-end RAG query: Hybrid Retrieval -> RRF -> Reranking -> Ollama Generation."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"QUERY: {query}")
        print(f"{'='*70}")

    # 1. Retrieve & Rerank Context Chunks
    context_chunks = hybrid_retriever.retrieve(
        query=query,
        top_k_dense=TOP_K_DENSE,
        top_k_sparse=TOP_K_SPARSE,
        top_k_final=TOP_K_RERANKED
    )

    # 2. Generate Grounded Answer via Ollama
    result = generator.generate_answer(query, context_chunks)
    result["retrieved_chunks"] = context_chunks

    return result

print("[OK] run_regulatory_rag_query function ready.")


[OK] run_regulatory_rag_query function ready.


### 🧪 Dedicated Test Query
Executes the query: `"What is the Cash Reserve Ratio requirement?"` through the entire pipeline and displays all required output blocks.


In [14]:
# ==============================================================================
# Cell 16: Dedicated Test Query Execution
# ==============================================================================
test_query = "What is the Cash Reserve Ratio requirement?"
result = run_regulatory_rag_query(test_query, verbose=False)

print("=== RETRIEVED CHUNKS ===")
for idx, chunk in enumerate(result.get("retrieved_chunks", []), start=1):
    meta = chunk.get("metadata", {})
    print(f"[Chunk {idx}] ID: {chunk.get('chunk_id')} | Doc: {meta.get('doc_title')} | Section: {meta.get('section')} | Score: {chunk.get('score', 0):.4f}")
    print(f"Snippet: {chunk.get('text', '')[:160]}...\n")

print("=== RERANKED CONTEXT ===")
for idx, chunk in enumerate(result.get("retrieved_chunks", []), start=1):
    meta = chunk.get("metadata", {})
    print(f"[Reranked #{idx}] Rerank Score: {chunk.get('rerank_score', 0.0):.4f} | Doc: {meta.get('doc_title')} ({meta.get('year')})")
    print(f"Section: {meta.get('section')}")
    print(f"Content: {chunk.get('text', '')[:200]}...\n")

print("=== GENERATED ANSWER ===")
print(result["answer"])

print("\n=== SOURCE CITATIONS ===")
for c in result.get("citations", []):
    print(f" - {c}")

print("\n=== DISCRETE CLAIMS ===")
for idx, claim in enumerate(result.get("claims", []), start=1):
    print(f" [{idx}] {claim}")

print("\n=== MODEL USED ===")
print(result["model_used"])


=== RETRIEVED CHUNKS ===
[Chunk 1] ID: RBI_Cash_Reserve_Ratio_(CRR)2009_chunk_3 | Doc: Cash Reserve Ratio (CRR)2009 | Section: 3. Annex | Score: 0.7397
Snippet: [Issuer: RBI | Category: Guideline | Year: 2009 | Doc: Cash Reserve Ratio (CRR)2009 | Chapter: General | Section: 3. Annex | Page: 3]

Master Circular  -  Cash ...

[Chunk 2] ID: RBI_MC163_180909_F_chunk_3 | Doc: MC163 180909 F | Section: 3. Annex | Score: 0.7392
Snippet: [Issuer: RBI | Category: Guideline | Year: 2009 | Doc: MC163 180909 F | Chapter: General | Section: 3. Annex | Page: 3]

Master Circular  -  Cash Reserve Ratio ...

[Chunk 3] ID: RBI_Cash_Reserve_Ratio_(CRR)2009_chunk_4 | Doc: Cash Reserve Ratio (CRR)2009 | Section: 1.4    Computation of Demand and Time Liabilities | Score: 7.0692
Snippet: [Issuer: RBI | Category: Guideline | Year: 2009 | Doc: Cash Reserve Ratio (CRR)2009 | Chapter: General | Section: 1.4    Computation of Demand and Time Liabilit...

[Chunk 4] ID: RBI_Cash_Reserve_Ratio_(CRR)2009_chunk_17 | D

### 📊 Automated Evaluation Framework
Replaces `python main.py eval` with an in-notebook benchmark evaluation suite over the benchmark dataset (`data/eval_dataset.json`).
Calculates:
- **Hit Rate@K**: Proportion of test queries where at least one ground-truth relevant document was retrieved in the top-$K$ reranked context.
- **Mean Reciprocal Rank (MRR@K)**: Average reciprocal rank of the first relevant retrieved document.
- **Precision@K**: Average proportion of retrieved top-$K$ chunks that are relevant.
- **Citation Coverage Rate**: Percentage of answers containing valid grounded citations.
- **Average Claims Generated**: Mean count of atomic assertions extracted per answer.


In [15]:
# ==============================================================================
# Cell 17: Benchmark Evaluator Class (RAGEvaluator)
# ==============================================================================
class RAGEvaluator:
    """Evaluation framework for measuring RAG retrieval accuracy and answer generation quality."""

    def __init__(self, retriever: HybridRerankRetriever, generator: RegulatoryGenerator):
        self.retriever = retriever
        self.generator = generator

    def evaluate_dataset(self, eval_dataset_path: Path = EVAL_DATASET_PATH, top_k: int = TOP_K_RERANKED) -> Dict[str, Any]:
        """Runs evaluation over ground-truth benchmark dataset."""
        eval_path = Path(eval_dataset_path)
        if not eval_path.exists():
            print(f"[Error] Evaluation dataset file not found: {eval_path}")
            return {}

        with open(eval_path, "r", encoding="utf-8") as f:
            test_cases = json.load(f)

        print(f"[Evaluator] Starting evaluation on {len(test_cases)} test cases (Top-K={top_k})...\n")

        total_reciprocal_rank = 0.0
        hits = 0
        total_precision = 0.0
        total_claims_generated = 0
        citation_count = 0

        eval_results = []

        for item in test_cases:
            qid = item["query_id"]
            query = item["query"]
            target_kw = item["target_doc_keyword"].lower()
            expected_kws = [k.lower() for k in item.get("expected_keywords", [])]

            # 1. Run Retrieval
            retrieved_chunks = self.retriever.retrieve(query, top_k_final=top_k)

            first_rank = 0
            relevant_retrieved = 0

            for rank, chunk in enumerate(retrieved_chunks, start=1):
                text_content = chunk.get("text", "").lower()
                doc_title = chunk.get("metadata", {}).get("doc_title", "").lower()

                # Check relevance matching
                is_relevant = (target_kw in doc_title or target_kw in text_content or
                               any(ek in text_content for ek in expected_kws))

                if is_relevant:
                    relevant_retrieved += 1
                    if first_rank == 0:
                        first_rank = rank

            prec = relevant_retrieved / top_k if top_k > 0 else 0.0
            rr = 1.0 / first_rank if first_rank > 0 else 0.0

            if first_rank > 0:
                hits += 1

            total_precision += prec
            total_reciprocal_rank += rr

            # 2. Run Answer Generation
            try:
                gen_output = self.generator.generate_answer(query, retrieved_chunks)
                claims = gen_output.get("claims", [])
                citations = gen_output.get("citations", [])
                model_used = gen_output.get("model_used", self.generator.model_name)
            except Exception as e:
                print(f"[Warning] Generation error for query '{qid}': {e}")
                claims = []
                citations = []
                model_used = f"Error: {e}"

            total_claims_generated += len(claims)
            if citations:
                citation_count += 1

            eval_results.append({
                "query_id": qid,
                "query": query,
                "first_hit_rank": first_rank,
                "precision_at_k": round(prec, 4),
                "mrr": round(rr, 4),
                "model_used": model_used,
                "citations_found": len(citations),
                "claims_extracted": len(claims)
            })

        num_samples = len(test_cases)
        mrr_score = total_reciprocal_rank / num_samples if num_samples > 0 else 0.0
        mean_precision = total_precision / num_samples if num_samples > 0 else 0.0
        hit_rate = hits / num_samples if num_samples > 0 else 0.0
        citation_rate = citation_count / num_samples if num_samples > 0 else 0.0

        summary = {
            "num_test_cases": num_samples,
            "top_k": top_k,
            "mean_precision_at_k": round(mean_precision, 4),
            "mrr": round(mrr_score, 4),
            "hit_rate": round(hit_rate, 4),
            "citation_coverage_rate": round(citation_rate, 4),
            "avg_claims_per_query": round(total_claims_generated / num_samples, 2) if num_samples > 0 else 0,
            "detailed_results": eval_results
        }

        print("=== EVALUATION SUMMARY ===")
        print(f"Hit Rate@{top_k}:           {summary['hit_rate'] * 100:.1f}%")
        print(f"Precision@{top_k}:          {summary['mean_precision_at_k']:.4f}")
        print(f"MRR@{top_k}:                {summary['mrr']:.4f}")
        print(f"Citation Coverage:     {summary['citation_coverage_rate'] * 100:.1f}%")
        print(f"Avg Claims/Query:      {summary['avg_claims_per_query']}")
        print("==========================\n")

        return summary

# Initialize Evaluator
evaluator = RAGEvaluator(retriever=hybrid_retriever, generator=generator)
print("[OK] RAGEvaluator defined.")


[OK] RAGEvaluator defined.


### 📈 Run Evaluation Benchmark & Export Summary Report
Executes the evaluation dataset, creates a formatted DataFrame of results, and exports the findings to `evaluation_report.json`.


In [16]:
# ==============================================================================
# Cell 18: Run Evaluation Benchmark & Display Results
# ==============================================================================
summary_results = evaluator.evaluate_dataset(EVAL_DATASET_PATH, top_k=TOP_K_RERANKED)

if summary_results and "detailed_results" in summary_results:
    # Display results in tabular format
    df_eval = pd.DataFrame(summary_results["detailed_results"])
    print("\nDetailed Evaluation Results:")
    try:
        from IPython.display import display
        display(df_eval)
    except ImportError:
        print(df_eval.to_string(index=False))

    # Save summary report to disk
    report_path = BASE_DIR / "evaluation_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(summary_results, f, indent=2)
    print(f"\n[OK] Evaluation report saved to {report_path}")


[Evaluator] Starting evaluation on 5 test cases (Top-K=5)...

=== EVALUATION SUMMARY ===
Hit Rate@5:           60.0%
Precision@5:          0.6000
MRR@5:                0.6000
Citation Coverage:     100.0%
Avg Claims/Query:      6.0


Detailed Evaluation Results:


,query_id,query,first_hit_rank,precision_at_k,mrr,model_used,citations_found,claims_extracted
0,q1,What are the Know Your Customer (KYC) guidelin...,1,1.0,1.0,llama3:latest,5,9
1,q2,What is the Cash Reserve Ratio (CRR) requireme...,1,1.0,1.0,llama3:latest,5,4
2,q3,What are the master circular guidelines for Me...,0,0.0,0.0,llama3:latest,5,4
3,q4,What regulations govern Alternative Investment...,0,0.0,0.0,llama3:latest,3,5
4,q5,What are the RBI guidelines regarding Lead Ban...,1,1.0,1.0,llama3:latest,5,8



[OK] Evaluation report saved to c:\Users\laksh\Downloads\FYP_RAG\evaluation_report.json
